# Live speech translator (notebook version)
Choose two of 12 languages by voice, then speak either one. Speech is recognized with Google Speech Recognition, translated by my fine-tuned mT5 model (Gunal13/mt5-small-11lang), and spoken with gTTS. The browser version is app.py.

## 1. Load the translation model

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "Gunal13/mt5-small-11lang"
mt5_tok = AutoTokenizer.from_pretrained("google/mt5-small")  # same tokenizer, older-compatible format
mt5_model, info = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID, tie_word_embeddings=False, output_loading_info=True)
mt5_model.set_input_embeddings(mt5_model.get_input_embeddings())  # needed on transformers 5.x; harmless on 4.x
mt5_model.eval()

print("Missing weights:   ", info["missing_keys"])
print("Unexpected weights:", info["unexpected_keys"])
print("Output table shared with input table?",
      mt5_model.lm_head.weight.data_ptr() == mt5_model.shared.weight.data_ptr())

def translate(text, src, dst):
    """Translate with my mT5 model. Non-English pairs go through English (pivot)."""
    if src == dst:
        return text
    if src != "English" and dst != "English":
        text = translate(text, src, "English")
        src = "English"
    prompt = f"translate {src} to {dst}: {text}"
    enc = mt5_tok(prompt, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        out = mt5_model.generate(**enc, max_new_tokens=128, num_beams=4,
                                 no_repeat_ngram_size=3)
    return mt5_tok.decode(out[0], skip_special_tokens=True)

tests = [
    ("How are you?", "English", "Hindi"),
    ("आप कैसे हो", "Hindi", "English"),
    ("Where is the train station?", "English", "Spanish"),
    ("Thank you very much.", "English", "Tamil"),
    ("आप कैसे हो", "Hindi", "Spanish"),   # pivot: Hindi → English → Spanish
]
for text, src, dst in tests:
    start = time.time()
    result = translate(text, src, dst)
    print(f"{src} → {dst}: {text}\n  → {result}   ({time.time() - start:.1f} s)\n")

c:\Users\gunal\Documents\voice-translation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
c:\Users\gunal\Documents\voice-translation\venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast token

Missing weights:    []
Unexpected weights: []
Output table shared with input table? False
English → Hindi: How are you?
  → तुम कैसे हो?   (1.2 s)

Hindi → English: आप कैसे हो
  → How are you?   (0.8 s)

English → Spanish: Where is the train station?
  → ¿Dónde está la estación de train?   (1.7 s)

English → Tamil: Thank you very much.
  → மிக்க நன்றி.   (0.9 s)

Hindi → Spanish: आप कैसे हो
  → ¿Cómo estás?   (2.0 s)



## 2. Live conversation

In [ ]:
import os
import tempfile
import uuid
import speech_recognition as sr
from gtts import gTTS
from playsound import playsound
MIC_INDEX = 2   # "AI Noise-cancelling Input" on my laptop; change for other computers

# name: (speech recognition code, text-to-speech code, script range for auto-detect)
LANGUAGES = {
    "English":   ("en-US", "en",    None),
    "Hindi":     ("hi-IN", "hi",    ("\u0900", "\u097F")),
    "Marathi":   ("mr-IN", "mr",    ("\u0900", "\u097F")),
    "Tamil":     ("ta-IN", "ta",    ("\u0B80", "\u0BFF")),
    "Kannada":   ("kn-IN", "kn",    ("\u0C80", "\u0CFF")),
    "Malayalam": ("ml-IN", "ml",    ("\u0D00", "\u0D7F")),
    "Telugu":    ("te-IN", "te",    ("\u0C00", "\u0C7F")),
    "Chinese":   ("zh-CN", "zh-CN", ("\u4E00", "\u9FFF")),
    "Arabic":    ("ar-SA", "ar",    ("\u0600", "\u06FF")),
    "Russian":   ("ru-RU", "ru",    ("\u0400", "\u04FF")),
    "Spanish":   ("es-ES", "es",    None),
    "Afrikaans": ("af-ZA", "af",    None),
}


def speak(text, lang):
    path = os.path.join(tempfile.gettempdir(), f"tts_{uuid.uuid4().hex}.mp3")
    gTTS(text=text, lang=LANGUAGES[lang][1]).save(path)
    playsound(path)
    try:
        os.remove(path)
    except PermissionError:
        pass   # Windows still holds the file; it's in the temp folder, so it's harmless


def matches_script(text, lang):
    """True if the text contains letters from that language's alphabet."""
    rng = LANGUAGES[lang][2]
    if rng is None:
        return True
    return any(rng[0] <= ch <= rng[1] for ch in text)


def recognize_best(r, audio, lang_a, lang_b):
    """Recognize the audio as both languages and keep the better match."""
    best = None
    for lang in (lang_a, lang_b):
        try:
            result = r.recognize_google(audio, language=LANGUAGES[lang][0], show_all=True)
        except sr.UnknownValueError:
            continue   # nothing recognized in this language, try the other one
        except sr.RequestError as e:
            print(f"Speech service error (check internet): {e}")
            continue
        if not result or "alternative" not in result:
            continue
        top = result["alternative"][0]
        text = top.get("transcript", "")
        score = top.get("confidence", 0.5)
        if not matches_script(text, lang):
            score -= 1.0   # e.g. a "Hindi" result written in English letters is a wrong guess
        if best is None or score > best[0]:
            best = (score, lang, text)
    return best


def choose_language(r, source, prompt):
    """Ask by voice until a supported language is named."""
    while True:
        print(prompt)
        speak(prompt, "English")
        try:
            heard = r.recognize_google(r.listen(source), language="en-US").lower()
        except sr.UnknownValueError:
            print("Couldn't understand, please try again.")
            continue
        for name in LANGUAGES:
            if name.lower() in heard:
                print(f"Selected: {name}")
                return name
        print(f"Heard '{heard}'. Supported: {', '.join(LANGUAGES)}")


r = sr.Recognizer()
r.pause_threshold = 0.8     # 0.8 s of silence = end of sentence
r.energy_threshold = 300

with sr.Microphone(device_index=MIC_INDEX) as source:
    print("Calibrating for background noise... stay quiet for a second.")
    r.adjust_for_ambient_noise(source, duration=1)

    speak("Welcome to Real Time Speech Translation.", "English")
    lang1 = choose_language(r, source, "Which language will you speak?")
    lang2 = choose_language(r, source, "Which language should I translate to?")

    msg = f"Ready. Speak in {lang1}, and I will translate to {lang2}."
    print(msg + "  (Press the stop button to end.)")
    speak(msg, "English")

    try:
        while True:
            print(f"\nListening... ({lang1} or {lang2})")
            audio = r.listen(source)
            best = recognize_best(r, audio, lang1, lang2)
            if best is None:
                print("Couldn't understand you")
                continue

            _, src, text = best
            dst = lang2 if src == lang1 else lang1
            translated = translate(text, src, dst)   # my mT5 model
            print(f"[{src}] {text}\n   → [{dst}] {translated}")
            speak(translated, dst)

    except KeyboardInterrupt:
        print("\nStopped.")
        for lang in (lang1, lang2):
            speak(translate("Thank you, have a nice day.", "English", lang), lang)

Calibrating for background noise... stay quiet for a second.
Which language will you speak?
Selected: Tamil
Which language should I translate to?
Selected: Hindi
Ready. Speak in Tamil, and I will translate to Hindi.  (Press the stop button to end.)

Listening... (Tamil or Hindi)
[Tamil] வணக்கம்
   → [Hindi] तुम्हारे लिए धन्यवाद.

Listening... (Tamil or Hindi)
Couldn't understand you

Listening... (Tamil or Hindi)
[Tamil] எப்படி இருக்க
   → [Hindi] यह कैसा है?

Listening... (Tamil or Hindi)
[Tamil] சாப்பிட்டியா
   → [Hindi] खाना

Listening... (Tamil or Hindi)

Stopped.
